<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/kol_network_stance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install

In [1]:
import importlib.util
import torch

!pip install -q "great[viz] @ git+https://github.com/azmkto/GREAT-Tools.git"
PACKAGES = ["emoji", "openpyxl", "ftfy", "networkx"]
PIP_ARGS = " ".join(PACKAGES)
!pip install -q -U {PIP_ARGS}

CU_TAG = "cu124" if torch.cuda.is_available() else "cpu"
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{CU_TAG}
if CU_TAG == "cpu":
    print("Minimal runtime type adalah T4 GPU.")

missing = [m for m in ["great", "tqdm", "huggingface_hub", "llama_cpp", *PACKAGES] if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Gagal memasang: {missing}")
print(f"(torch CUDA {torch.version.cuda}, llama-cpp {CU_TAG}).")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 845.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
(torch CUDA 13.0, llama-cpp cu124).


# 2. Import Library

In [2]:
import json, os, re, sys, time, unicodedata   # json: baca/simpan config, os: cek file, re: regex, sys: info modul, time: jeda antar request, unicodedata: rapiin huruf
from collections import Counter     # ngitung keyword yang paling sering lolos
from datetime import date           # tanggal hari ini buat prompt config topik
from functools import lru_cache     # simpen hasil canonical() biar nama yang sama nggak dihitung ulang

import emoji                         # ubah emoji jadi teks
import networkx as nx                # bikin jaringan akun, hitung pagerank
import numpy as np                   # hitung eigenvector
import pandas as pd                  # olah tabel data
import torch                         # cek ada GPU atau nggak
from google.colab import ai, drive   # ai: gemini, drive: simpen config di google drive
from IPython.display import display  # tampilin tabel di notebook
from tqdm.auto import tqdm           # progress bar

from great import EXPORT_COLUMNS, clean_for_bert, is_media_account   # dari GREAT-Tools
from great.labels import is_media_name                              # cek akun media dari nama kanal

tqdm.pandas()   # biar nampilin progress bar

print("great dari:", sys.modules["great"].__file__)
print("Model Colab AI:", ai.list_models())

great dari: /usr/local/lib/python3.13/dist-packages/great/__init__.py
Model Colab AI: ['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-3.5-flash', 'google/gemini-3.5-flash-lite', 'google/gemini-3.7-flash']


# Colab AI

In [3]:
AI_MODEL = "google/gemini-2.5-flash"   # milih model AI, subject to availability
SLEEP_BETWEEN_CALLS = 4                # jeda antar request biar nggak kena limit
api_call_count = 0


def is_quota_error(e):
    msg = str(e).lower()
    return "429" in msg or "resource_exhausted" in msg or "quota" in msg


def parse_json_text(text):
    # ngambil JSON pertama dari jawaban gemini
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    if not starts:
        raise ValueError("jawaban model nggak ada JSON-nya")
    return json.JSONDecoder().raw_decode(text[min(starts):])[0]


def ask_json(prompt, parse=lambda x: x, retry=4):
    # kirim prompt ke gemini, balikin JSON. kuota habis langsung stop.
    global api_call_count
    for attempt in range(retry):
        try:
            api_call_count += 1
            return parse(parse_json_text(ai.generate_text(prompt, model_name=AI_MODEL)))
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = (20 if "503" in str(e) or "UNAVAILABLE" in str(e) else 5) * (attempt + 1)
            print(f"AI gagal ({attempt + 1}/{retry}): {e}, coba lagi {wait} detik")
            time.sleep(wait)
    return None


def order_by_index(parsed, n):
    # urutin hasil sesuai field index 1..n, error kalau ada yang ilang
    by_idx = {int(it["index"]): it for it in parsed if isinstance(it, dict) and "index" in it}
    if set(by_idx) != set(range(1, n + 1)):
        raise ValueError(f"index dari AI nggak lengkap: dapet {sorted(by_idx)}, harusnya 1..{n}")
    return [by_idx[i] for i in range(1, n + 1)]

# 3. Form

In [4]:
PROJECT = "prabowo_vladivostok"   # nama project, dipake buat nama file config & checkpoint di Drive
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"   # topik yang dianalisis, dibaca AI buat bikin tema & keyword
STANCE_TARGET = "Presiden Prabowo dan pemerintah"   # pihak yang dinilai stance-nya (supporter / moderat / oposisi)
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"   # link data Google Sheets/Drive, kalau lebih dari satu pisahin pake koma
REGENERATE_TOPIC_CONFIG = False   # True = AI bikin ulang tema & keyword (misal kalau keyword-nya terlalu umum), False = pake config yang udah kesimpen

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC = TOPIC.strip()
STANCE_TARGET = STANCE_TARGET.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]

problems = [msg for ok, msg in [(PROJECT, "PROJECT masih kosong"),
                                (TOPIC, "TOPIC masih kosong"),
                                (STANCE_TARGET, "STANCE_TARGET masih kosong"),
                                (SOURCES, "SOURCES_TEXT masih kosong, isi link Google Sheets")] if not ok]
if problems:
    raise ValueError("Form belum lengkap: " + "; ".join(problems))
print(f"Project: {PROJECT}")
print(f"Topik: {TOPIC}")
print(f"Stance dinilai terhadap: {STANCE_TARGET}")
print(f"Jumlah file: {len(SOURCES)}")

Project: prabowo_vladivostok
Topik: Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin
Stance dinilai terhadap: Presiden Prabowo dan pemerintah
Jumlah file: 1


# 4. Konfigurasi

In [5]:
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]   # Location nggak dipake

# LLM lokal buat sentimen, stance, summary (Step 6-7), sama kayak topic_modelling.ipynb
LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192

# bobot skor volume, totalnya 1
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15
# bobot skor jaringan, totalnya 1
NET_WEIGHT_PAGERANK = 0.7   # sering di-RT/mention akun yang juga berpengaruh
NET_WEIGHT_EIGEN = 0.3      # nyambung ke akun-akun yang juga rame koneksinya
# skor KOL akhir = volume + jaringan, totalnya 1
KOL_WEIGHT_VOLUME = 0.5    # post asli + engagement + followers
KOL_WEIGHT_NETWORK = 0.5   # pagerank + eigenvector

BATCH_SIZE_CLASSIFY = 40     # cuitan per request gemini di Step 4
BATCH_SIZE_VALIDATE = 5      # KOL per request gemini di Step 6C
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

RANKING_MIN_CONFIDENCE = 0.8      # tema yang confidence-nya di bawah ini nggak ikut ranking

TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
# hasil nggak di-export, cukup ditampilin di notebook
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Mounted at /content/drive


# Konfigurasi topik dari Colab AI

In [7]:
TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia yang menyiapkan konfigurasi klasifikasi otomatis.
Topik yang dipantau: "{TOPIC}".
Tanggal hari ini: {date.today()}. Jangan menulis tahun di keyword kecuali yakin tahunnya benar.

Konfigurasi ini dipakai oleh pipeline berikut, jadi ikuti aturan tiap field dengan ketat:
1. Cuitan yang TIDAK mengandung satu pun `keywords` langsung dibuang sebagai "lainnya" tanpa dibaca AI.
   Keyword yang terlewat = cuitan relevan hilang diam-diam. Keyword terlalu umum = ribuan cuitan
   tidak relevan ikut diproses (lama & boros kuota).
2. Cuitan yang lolos dibaca AI dan diberi satu tema berdasarkan deskripsi di `themes`.
3. Akun di `media_accounts` ditandai sebagai akun media.

PENTING: data mentah biasanya hasil monitoring yang LEBIH LUAS dari topik ini, misalnya SEMUA cuitan
yang menyebut tokoh utamanya (termasuk isu lain yang tidak berhubungan). Karena itu nama tokoh utama
saja BUKAN penanda topik ini.

=== ATURAN PER FIELD ===

themes (objek: nama_tema -> deskripsi)
- 2-4 tema yang membagi percakapan tentang topik ini secara jelas. Tiap cuitan relevan harus
  masuk tepat satu tema. Pilih pembagian berdasarkan sudut pandang percakapan (misal: agenda/acara,
  isu kebijakan/kerja sama, reaksi politik/geopolitik), bukan berdasarkan tokoh.
- Nama: snake_case, singkat, bahasa Indonesia (contoh: kerja_sama_ekonomi).
- Deskripsi satu string dengan format:
  "Masuk: <apa yang dibahas, 1 kalimat>. Contoh frasa: <8-15 frasa khas cuitan, termasuk bahasa
  gaul, singkatan, julukan>. Bukan: <yang sering tertukar dengan tema lain dan masuk ke tema mana>."
- Jangan buat tema "lainnya", "umum", atau "campuran". Itu sudah disediakan sistem.

keywords (list string)
- 20-60 kata/frasa, huruf kecil semua, dicocokkan sebagai frasa PERSIS dan KATA UTUH.
  Contoh: "putin" TIDAK cocok dengan "putinisme"; "prabowo putin" TIDAK cocok dengan "prabowo bertemu putin".
  Jadi utamakan kata tunggal yang khas, bukan frasa panjang.
- Isi dengan penanda yang KHAS topik ini: nama pihak lain, tempat, acara, dan instansi yang hampir
  hanya muncul saat membicarakan topik ini (ejaan Indonesia dan Inggris, singkatan, salah ketik umum,
  bentuk hashtag digabung tanpa #).
- DILARANG: nama tokoh utama sendirian (misal "prabowo", "presiden prabowo", julukannya), dan kata umum
  yang juga dipakai di isu lain (misal "investasi", "pertahanan", "kesepakatan", "perdamaian",
  "kunjungan kerja", "kebijakan luar negeri", "ketahanan pangan").

media_accounts (list string)
- Username akun berita/media dan kanal resmi institusi yang khusus terkait topik ini
  (contoh: kantor berita negara asing, akun humas lembaga). Huruf kecil, tanpa @.
- JANGAN masukkan akun pribadi tokoh politik, buzzer, atau influencer: mereka justru
  harus ikut dianalisis sebagai author.
- Kalau tidak yakin username-nya ada, jangan dimasukkan. List kosong boleh.

=== CEK SEBELUM MENJAWAB ===
- Tidak ada keyword berupa nama tokoh utama sendirian atau kata umum.
- Setiap tema punya bagian "Bukan:" yang menyebut batas dengan tema lain.
- Semua keywords huruf kecil.

=== FORMAT JAWABAN ===
Jawab HANYA satu objek JSON, tanpa markdown, tanpa ```, tanpa teks sebelum atau sesudahnya:
{{"themes": {{"nama_tema": "Masuk: ... Contoh frasa: ... Bukan: ..."}},
  "keywords": ["...", "..."],
  "media_accounts": ["..."]}}"""

# config lama cuma dipake kalau topiknya sama persis
topic_config = None
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        cached = json.load(f)
    if cached.get("_topic") == TOPIC:
        topic_config = cached
        print(f"Config topik dibaca dari {TOPIC_CONFIG_PATH}")
    else:
        print("topik beda sama config yang kesimpen, AI bikin ulang")

if topic_config is None:
    topic_config = ask_json(TOPIC_CONFIG_PROMPT)
    if not topic_config or not topic_config.get("themes"):
        raise RuntimeError("Colab AI gagal bikin config topik. Jalanin ulang cell ini")
    topic_config["_topic"] = TOPIC
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)
    print(f"Config topik dari AI disimpan ke {TOPIC_CONFIG_PATH}")

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = [t for t in THEMES if t != "lainnya"]
KW_RELEVAN = [k.lower() for k in topic_config.get("keywords", [])]
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config.get("media_accounts", [])}

# cuitan tanpa keyword sama sekali langsung 'lainnya'. kalau keyword kosong, semua cuitan ke AI
RE_RELEVAN = re.compile(
    r"(?<!\w)(?:" + "|".join(map(re.escape, sorted(set(KW_RELEVAN), key=len, reverse=True))) + r")(?!\w)"
) if KW_RELEVAN else None


print(json.dumps(topic_config, ensure_ascii=False, indent=2))

Config topik dibaca dari /content/drive/MyDrive/topic_config_prabowo_vladivostok.json
{
  "themes": {
    "agenda_pertemuan": "Masuk: Pembahasan detail terkait jadwal, lokasi, susunan acara, dan siapa saja yang terlibat dalam kunjungan Presiden Prabowo ke Vladivostok serta pertemuannya dengan Vladimir Putin. Contoh frasa: tiba vladivostok, agenda prabowo putin, pertemuan bilateral, kunjungan kenegaraan, delegasi indonesia, jadwal presiden, sambutan hangat, jam berapa. Bukan: Hasil perjanjian atau kerja sama (masuk kerja_sama_bilateral), atau tanggapan/analisis politis (masuk reaksi_geopolitik).",
    "kerja_sama_bilateral": "Masuk: Diskusi mengenai potensi kesepakatan, MoU, atau area kerja sama spesifik antara Indonesia dan Rusia, termasuk sektor pertahanan, energi, ekonomi, pangan, dan teknologi. Contoh frasa: kesepakatan pertahanan, kerjasama energi, investasi rusia, MoU indonesia rusia, pangan, alutsista, perdagangan bilateral, ekonomi strategis, pasokan minyak, swasembada. Bukan: D

# Load Data dari Drive


In [8]:
def xlsx_url(link):
    # link google sheets/drive -> link download xlsx
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastiin aksesnya 'Anyone with the link'") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # kalau ada kolom yang ilang bakal KeyError

df_raw = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"Data ke-load: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

Data ke-load: 66891 baris, 13 kolom


# Filter akun media


In [9]:
# satu akun bisa muncul pake beberapa ejaan, disatuin jadi satu key
ALIASES = {"kompastvcom": "kompascom"}   # buat gabungin manual
TWO_PART_SLD = {"co", "or", "ac", "go", "web", "my", "sch", "net", "com"}


@lru_cache(maxsize=None)   # nama author yang sama dipanggil berkali-kali; abis edit ALIASES jalanin ulang cell ini
def canonical(name):
    if pd.isna(name):
        return None
    s = re.sub(r"^(https?://)?(www\.)?@?", "", unicodedata.normalize("NFKC", str(name)).strip().lower()).split("/")[0]
    parts = s.split(".")
    if len(parts) > 2 and re.fullmatch(r"[a-z]{2,3}", parts[-1]):   # domain, bukan handle IG yang ada titiknya
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r"[^a-z0-9]", "", "".join(parts)) or re.sub(r"\W", "", s)   # nama non-latin tetep pake hurufnya
    return ALIASES.get(key, key) if key else None                           # kosong / emoji doang = bukan akun


assert canonical("@KOMPASTVCOM") == canonical("www.kompastv.com")   # ejaan beda, akun sama
assert canonical("nasional.kompas.com") == canonical("@kompascom") == "kompascom"
assert canonical("https://www.republika.co.id/berita/x") == "republikacoid"
assert canonical("prabowo.subianto") == canonical("@Prabowo_Subianto") == "prabowosubianto"
assert canonical("𝐁𝐞𝐫𝐢𝐭𝐚 𝐈𝐧𝐝𝐨") == "beritaindo" and canonical("بيت") == "بيت" and canonical("🔥🔥") is None
assert canonical(float("nan")) is None and canonical("  ") is None

In [10]:
INCLUDE_MEDIA = False   # True = akun media ikut dianalisis & bisa jadi KOL, False = akun media dibuang

NUM_COLS = ["Followers", "Retweeted", "Favourited"]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

by_account = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                        for a, p in zip(df_raw["Author"], df_raw["Media"])], index=df_raw.index)
by_name = df_raw["Author"].map(is_media_name)   # nama kanal YouTube/Facebook yang ada spasinya
is_media = by_account | by_name

# media yang ketauan dari satu ejaan (www.kompastv.com) ikut ditandain di ejaan lain (@KOMPASTVCOM).
# MEDIA_IDS sama author_key dipake lagi di Step 5
df_raw["author_key"] = df_raw["Author"].map(canonical)
MEDIA_IDS = set(df_raw.loc[is_media, "author_key"].dropna()) | {canonical(a) for a in EXTRA_MEDIA_ACCOUNTS}
by_merge = df_raw["author_key"].isin(MEDIA_IDS) & ~is_media
is_media |= by_merge
df_raw["is_media"] = is_media
print(f"Baris media: {is_media.sum()} ({(by_name & ~by_account).sum()} dari nama kanal, {by_merge.sum()} dari ejaan lain)")

if INCLUDE_MEDIA:
    print("Akun media ikut dianalisis, Step 4 bakal lebih lama & boros kuota")
else:
    df_raw = df_raw[~is_media].reset_index(drop=True)
    print("Akun media dibuang")
print(f"Sisa {len(df_raw)} baris, {df_raw['author_key'].isna().sum()} di antaranya tanpa Author (tetep dipake buat tema)")

Baris media: 29558 (790 dari nama kanal, 77 dari ejaan lain)
Akun media dibuang
Sisa 37333 baris, 11478 di antaranya tanpa Author (tetep dipake buat tema)


# Cleaning text

In [11]:
# di export ini retweet ditulis "RT <teks>" (sering tanpa @user), balesan ditulis "... [RE username]"
RE_RT_PREFIX = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")      # "RT" di awal = retweet
RE_RT = re.compile(r"(?<!\w)RT\s*@(\w+)", re.IGNORECASE)        # akun yang di-RT, kalau ditulis
RE_REPLY = re.compile(r"\[RE\s+@?(\w+)\]", re.IGNORECASE)        # akun yang dibales
RE_ELONGATION = re.compile(r"(.)\1{2,}")
EMOJI_LANG = "id" if "id" in emoji.LANGUAGES else "en"


def strip_noise(text):
    # buang RT & [RE user], emoji jadi kata, 'mantaaap' -> 'mantaap'
    text = RE_REPLY.sub(" ", RE_RT_PREFIX.sub("", text))
    text = emoji.demojize(text, language=EMOJI_LANG).replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


assert strip_noise("RT @abc: halo [RE def]").strip() == "halo"
assert strip_noise("RT halo").strip() == "halo"


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


def dedup_key(texts):
    # kunci buat nyari duplikat (huruf kecil, tanpa tanda baca). dipake lagi di Step 5B
    return texts.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")


df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_raw["Headline"], df_raw["Mentions"])]
df_raw["is_retweet"] = df_raw["text_raw_combined"].map(lambda t: bool(RE_RT_PREFIX.match(t)))
df_raw["text_clean"] = df_raw["text_raw_combined"].map(strip_noise).progress_apply(clean_for_bert)

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3].copy()
print(f"Buang {len(df_raw) - len(df_clean)} baris kosong / kependekan")

before = len(df_clean)
df_clean = (df_clean.assign(dedup_key=dedup_key(df_clean["text_clean"]))
            .sort_values("is_retweet", kind="stable")   # post asli didahuluin biar yang kesimpen punya penulis aslinya
            .drop_duplicates(subset="dedup_key")
            .sort_index()
            .drop(columns="dedup_key").reset_index(drop=True))
print(f"Retweet: {df_raw['is_retweet'].sum()}, duplikat dibuang: {before - len(df_clean)}, sisa {len(df_clean)} baris")

  0%|          | 0/37333 [00:00<?, ?it/s]

Buang 1709 baris kosong / kependekan
Retweet: 13816, duplikat dibuang: 14164, sisa 21460 baris


# Klasifikasi tema (prefilter keyword + Colab AI)

In [12]:
THEME_LIST_STR = "\n".join(f"- {t}: {d}" for t, d in THEME_DESCRIPTIONS.items())


def build_theme_prompt(texts):
    # tiap cuitan dipotong 120 kata, artikel panjang nggak perlu dibaca full buat nentuin tema
    short = [" ".join(str(t).split()[:120]) for t in texts]
    joined = "\n".join(f'{i}. "{t}"' for i, t in enumerate(short, 1))
    return f"""Kamu classifier tema untuk cuitan bahasa Indonesia tentang: {TOPIC}.

Daftar tema:
{THEME_LIST_STR}

Tentukan satu tema untuk setiap cuitan.
- Tema "lainnya" WAJIB dipakai kalau cuitan tidak membahas topik di atas secara langsung
  (misal hanya menyebut nama tokoh, negara, atau acara yang sama untuk peristiwa lain).

Cuitan:
{joined}

Jawab HANYA JSON array berisi tepat {len(texts)} objek dengan field:
index (nomor cuitan), primary_theme (salah satu nama tema di atas), confidence (0-1)."""


df_clean[["theme", "theme_confidence"]] = None

# tahap 1: cuitan tanpa keyword langsung 'lainnya'
if RE_RELEVAN:
    text_lower = df_clean["text_clean"].str.lower()
    irrelevant = ~text_lower.str.contains(RE_RELEVAN)
    df_clean.loc[irrelevant, ["theme", "theme_confidence"]] = ["lainnya", 1.0]

    kw_hits = Counter(k for t in text_lower[~irrelevant] for k in set(RE_RELEVAN.findall(t)))
    print("Keyword yang paling banyak lolos:", kw_hits.most_common(10))
    if (~irrelevant).mean() > 0.5:
        print(f"{(~irrelevant).mean():.0%} baris lolos prefilter, kayaknya ada keyword yang terlalu umum. "
              "Biar nggak lama & boros kuota, set REGENERATE_TOPIC_CONFIG = True di cell 2 terus jalanin ulang")
review_idx = df_clean.index[df_clean["theme"].isna()].tolist()
print(f"Prefilter: {len(df_clean) - len(review_idx)} baris langsung 'lainnya', {len(review_idx)} baris lanjut ke AI")

# tahap 2: gemini nentuin tema, satu batch per request (paralel langsung kena limit)
for b in tqdm(range(0, len(review_idx), BATCH_SIZE_CLASSIFY), desc="Klasifikasi tema"):
    idx_batch = review_idx[b:b + BATCH_SIZE_CLASSIFY]
    texts = df_clean.loc[idx_batch, "text_clean"].tolist()
    try:
        # hasil dicocokin pake field 'index', bukan urutan
        results = ask_json(build_theme_prompt(texts), parse=lambda p: order_by_index(p, len(texts)), retry=6)
    except Exception as e:
        if not is_quota_error(e):
            raise
        print("Kuota habis. Tunggu beberapa menit terus jalanin ulang cell ini")
        break
    if results is None:
        print(f"1 batch ({len(idx_batch)} cuitan) gagal, dibiarin tanpa tema")
    else:
        for r, df_idx in zip(results, idx_batch):
            theme = r.get("primary_theme")
            df_clean.at[df_idx, "theme"] = theme if theme in THEMES else "lainnya"
            df_clean.at[df_idx, "theme_confidence"] = float(r.get("confidence", 0.5))
    time.sleep(SLEEP_BETWEEN_CALLS)

n_unprocessed = df_clean["theme"].isna().sum()
if n_unprocessed:
    print(f"Masih ada {n_unprocessed} baris tanpa tema (kuota habis / batch gagal), hasil di bawah belum lengkap")
print("Distribusi tema:")
print(df_clean["theme"].value_counts(dropna=False).to_string())
print(f"Total request AI: {api_call_count}")

Keyword yang paling banyak lolos: [('rusia', 126), ('geopolitik', 103), ('infrastruktur', 103), ('komoditas', 83), ('brics', 69), ('ukraina', 29), ('alutsista', 25), ('kesepakatan', 16), ('digitalisasi', 14), ('vladivostok', 12)]
Prefilter: 20940 baris langsung 'lainnya', 520 baris lanjut ke AI


Klasifikasi tema:   0%|          | 0/13 [00:00<?, ?it/s]

Kuota habis. Tunggu beberapa menit terus jalanin ulang cell ini
Masih ada 280 baris tanpa tema (kuota habis / batch gagal), hasil di bawah belum lengkap
Distribusi tema:
theme
lainnya                 21163
None                      280
reaksi_geopolitik           8
kerja_sama_bilateral        6
agenda_pertemuan            3
Total request AI: 7


# Fungsi jaringan

In [13]:
HANDLE = re.compile(r"(?<![\w.])@([\w.]*\w)")   # handle IG yang ada titiknya ikut


def is_media_id(key):
    # akun media
    return key in MEDIA_IDS or is_media_account(key, extra_accounts=EXTRA_MEDIA_ACCOUNTS)


def extract_targets(text, retweet_of=None):
    # satu post -> satu edge per akun tujuan. prioritas: retweet > reply > mention
    # retweet_of = penulis asli buat retweet yang nggak nulis @user (dicari dari teks yang sama)
    rt = RE_RT.search(text)
    source = canonical(rt.group(1)) if rt else retweet_of
    targets = {source: "retweet"} if source else {}
    for m in RE_REPLY.findall(text):
        targets.setdefault(canonical(m), "reply")
    for m in HANDLE.findall(text):
        targets.setdefault(canonical(m), "mention")
    return list(targets.items())


def eigenvector_all(U):
    # eigenvector dihitung di tiap klaster (bukan cuma klaster terbesar), jadi semua akun dapet nilai.
    # tiap klaster dikali (nilai eigen klaster / nilai eigen terbesar), jadi klaster kecil nilainya ikut kecil
    parts = []
    for comp in nx.connected_components(U):
        nodes = list(comp)
        # eigh biasa cukup cepet sampe ribuan akun per klaster. kalau lebih gede, ganti scipy.sparse.linalg.eigsh
        vals, vecs = np.linalg.eigh(nx.to_numpy_array(U, nodelist=nodes, weight="Weight"))
        parts.append((vals[-1], pd.Series(np.abs(vecs[:, -1]), index=nodes)))
    lam_max = max(lam for lam, _ in parts)
    return pd.concat([vec * lam / lam_max for lam, vec in parts])


def centrality(pair_w):
    # pagerank (graf berarah) + eigenvector (graf nggak berarah) per akun
    G = nx.from_pandas_edgelist(pair_w, "Source", "Target", edge_attr="Weight", create_using=nx.DiGraph)
    pair = np.sort(pair_w[["Source", "Target"]].to_numpy(), axis=1)   # A->B sama B->A dijumlahin
    und = pair_w.assign(u=pair[:, 0], v=pair[:, 1]).groupby(["u", "v"], as_index=False)["Weight"].sum()
    U = nx.from_pandas_edgelist(und, "u", "v", edge_attr="Weight")
    out = pd.DataFrame({"pagerank": pd.Series(nx.pagerank(G, weight="Weight"))})
    out["eigenvector"] = eigenvector_all(U).reindex(out.index, fill_value=0.0)
    return out.rename_axis("Id").reset_index()


assert extract_targets("RT @Abc: halo @Def @abc. @def, email x@gmail.com") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("RT halo @def", retweet_of="abc") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("setuju banget [RE Abc]") == [("abc", "reply")]
assert extract_targets("cc @prabowo.subianto @Prabowo_Subianto") == [("prabowosubianto", "mention")]
# contoh kecil: 8 akun nge-reply "hub" (klaster gede) + 2 akun saling mention sekali (klaster kecil)
_toy = centrality(pd.DataFrame({"Source": [f"r{i}" for i in range(8)] + ["x"],
                                "Target": ["hub"] * 8 + ["y"], "Weight": [1] * 9})).set_index("Id")["eigenvector"]
assert abs(_toy["hub"] - 0.7071) < 1e-3 and abs(_toy["r0"] - 0.25) < 1e-3     # sama kayak versi klaster terbesar doang
assert 0 < _toy["x"] < _toy["hub"]                                             # klaster kecil nggak 0, tapi di bawah pusat klaster gede
assert abs(centrality(pd.DataFrame({"Source": ["a", "b"], "Target": ["b", "c"], "Weight": [1, 1]}))["pagerank"].sum() - 1) < 1e-6

# Edges

In [14]:
# cuitan yang ikut ranking: temanya relevan & AI cukup yakin
relevant = df_clean["theme"].isin(RELEVANT_THEMES)
confident = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE
df_topic = df_clean[relevant & confident].copy()
print(f"Baris yang ikut ranking: {len(df_topic)}, dibuang karena tema ragu (< {RANKING_MIN_CONFIDENCE}): "
      f"{(relevant & ~confident).sum()}")

author_key = df_raw["author_key"]
n_merged = df_raw.groupby("author_key")["Author"].nunique().gt(1).sum()
print(f"{df_raw['Author'].nunique():,} ejaan Author jadi {author_key.nunique():,} akun "
      f"({n_merged:,} akun hasil gabungan 2+ ejaan, kalau ada yang salah gabung benerin lewat ALIASES)")

# edges diambil dari df_raw (sebelum dedup) biar semua retweet kehitung
raw_keys = dedup_key(df_raw["text_clean"])
# tema ikut df_topic, jadi retweet ikut tema post aslinya
theme_by_key = dict(zip(dedup_key(df_topic["text_clean"]), df_topic["theme"]))
raw_theme = raw_keys.map(theme_by_key)
# "RT <teks>" tanpa @user: penulis aslinya = author post asli yang teksnya sama
orig_author = (pd.DataFrame({"key": raw_keys, "author": author_key})[~df_raw["is_retweet"]]
               .drop_duplicates("key").set_index("key")["author"])
retweet_of = raw_keys.map(orig_author).where(df_raw["is_retweet"])
print(f"Retweet: {df_raw['is_retweet'].sum()}, yang ketemu post aslinya: {retweet_of.notna().sum()}")

edge_rows = pd.DataFrame([
    {"Source": author, "Target": target, "interaction": kind, "theme": theme, "text": text}
    for author, raw, text, theme, rt_of in zip(author_key, df_raw["text_raw_combined"],
                                               df_raw["text_clean"], raw_theme, retweet_of)
    if pd.notna(theme) and author is not None   # post tanpa Author nggak punya asal edge
    for target, kind in extract_targets(raw, rt_of if pd.notna(rt_of) else None)
    if target != author
], columns=["Source", "Target", "interaction", "theme", "text"])
df_edges = edge_rows.groupby(["Source", "Target", "interaction", "theme"]).size().rename("Weight").reset_index()

# teks yang nge-RT X = kata-kata X sendiri, dipake di Step 6B buat KOL yang nggak posting
rt_posts = (edge_rows[edge_rows["interaction"] == "retweet"]
            .groupby(["Target", "theme"])["text"].apply(lambda s: s.drop_duplicates().tolist()).to_dict())
print(f"Edges: {len(df_edges)} ({', '.join(f'{k} {v}' for k, v in df_edges.groupby('interaction')['Weight'].sum().items())})")

Baris yang ikut ranking: 17, dibuang karena tema ragu (< 0.8): 0
17,217 ejaan Author jadi 17,146 akun (67 akun hasil gabungan 2+ ejaan, kalau ada yang salah gabung benerin lewat ALIASES)
Retweet: 13816, yang ketemu post aslinya: 4043
Edges: 6 (mention 1, reply 2, retweet 3)


# Skor KOL + Nodes

In [15]:
# skor volume: retweet bukan tulisan author-nya (kolom Retweeted-nya juga punya tweet asli), jadi nggak dihitung.
# pengaruhnya masuk ke penulis asli lewat jaringan
df_topic["engagement"] = df_topic["Favourited"] + df_topic["Retweeted"]
df_original = df_topic[~df_topic["is_retweet"]]

# dikelompokin per akun (bukan per ejaan), tiap metrik dibagi max-nya di tema yang sama
agg = (df_original.groupby(["theme", "author_key"])
       .agg(jumlah_post=("engagement", "size"), total_engagement=("engagement", "sum"), followers=("Followers", "max"))
       .reset_index().rename(columns={"author_key": "Id"}))
metrics = agg[["jumlah_post", "total_engagement", "followers"]].astype(float)
norm = metrics / metrics.groupby(agg["theme"]).transform("max").replace(0, 1)
agg["volume_score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]

# skor jaringan: pagerank + eigenvector per tema
net_parts = [centrality(sub.groupby(["Source", "Target"], as_index=False)["Weight"].sum()).assign(theme=theme)
             for theme, sub in df_edges.groupby("theme")]
df_net = (pd.concat(net_parts, ignore_index=True) if net_parts
          else pd.DataFrame(columns=["Id", "pagerank", "eigenvector", "theme"]))

# ejaan Author yang paling sering muncul per akun, buat label di tabel
labels = (df_raw.groupby(["author_key", "Author"]).size().sort_values(ascending=False)
          .reset_index().drop_duplicates("author_key").set_index("author_key")["Author"])
followers_by_id = df_raw.groupby("author_key")["Followers"].max()

kol = agg.merge(df_net, on=["theme", "Id"], how="outer")
# kalau INCLUDE_MEDIA = False, akun media tetep jadi node tapi nggak boleh jadi KOL
if not INCLUDE_MEDIA:
    kol = kol[~kol["Id"].map(is_media_id)]
kol = kol.copy()
kol["author_id"] = kol["Id"].map(labels).fillna(kol["Id"])   # akun yang cuma di-mention nggak punya ejaan Author
kol[["volume_score", "pagerank", "eigenvector"]] = kol[["volume_score", "pagerank", "eigenvector"]].fillna(0)
# dibagi max per tema setelah filter media, biar skornya sebanding
norm_theme = lambda col: kol[col] / kol.groupby("theme")[col].transform("max").replace(0, 1)
kol["network_score"] = NET_WEIGHT_PAGERANK * norm_theme("pagerank") + NET_WEIGHT_EIGEN * norm_theme("eigenvector")
kol["score"] = KOL_WEIGHT_VOLUME * kol["volume_score"] + KOL_WEIGHT_NETWORK * kol["network_score"]

df_top_authors = (kol.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

# nodes = semua akun di jaringan per tema, angkanya sama kayak di skor KOL
top_pairs = set(zip(df_top_authors["theme"], df_top_authors["Id"]))
df_nodes = df_net.assign(akun=df_net["Id"].map(labels).fillna(df_net["Id"]),
                         followers=df_net["Id"].map(followers_by_id),
                         is_kol=[p in top_pairs for p in zip(df_net["theme"], df_net["Id"])])
print(f"KOL: {len(df_top_authors)}, nodes: {len(df_nodes)}, edges: {len(df_edges)}")

# cek kolom Author isinya username, kalau bukan jaringannya nggak nyambung ke skor volume
if len(df_edges):
    match_rate = df_edges["Target"].isin(followers_by_id.index).mean()
    print(f"{match_rate:.0%} target edge juga muncul sebagai Author. "
          "Kalau mendekati 0%, cek kolom Author, harusnya username bukan nama tampilan")

KOL: 18, nodes: 8, edges: 6
67% target edge juga muncul sebagai Author. Kalau mendekati 0%, cek kolom Author, harusnya username bukan nama tampilan


#Load LLM lokal

In [16]:
import ctypes, glob, site
CUDA12_LIBS = ["libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"]


def find_lib(name):
    roots = {*site.getsitepackages(), *[p for p in sys.path if os.path.isdir(p)]}
    hits = [p for r in roots for p in glob.glob(f"{r}/nvidia/**/{name}", recursive=True)]
    return hits[0] if hits else None


if torch.cuda.is_available():
    if not all(map(find_lib, CUDA12_LIBS)):
        !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    missing_libs = [n for n in CUDA12_LIBS if not find_lib(n)]
    if missing_libs:
        raise RuntimeError(f"Library CUDA 12 tetep nggak ketemu: {missing_libs}")
    for name in CUDA12_LIBS:
        ctypes.CDLL(find_lib(name), mode=ctypes.RTLD_GLOBAL)

# import yang gagal sebelumnya bisa nyangkut, dibuang dulu biar import ulang dari awal
for m in [m for m in sys.modules if m.startswith("llama_cpp")]:
    del sys.modules[m]

import llama_cpp
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

if not torch.cuda.is_available():
    print("Ganti ke T4 GPU terus mulai lagi dari cell Install")
elif not llama_cpp.llama_supports_gpu_offload():
    print("llama-cpp ke-install tanpa GPU")

if "llm" in globals():
    print(f"LLM lokal udah ke-load, dipake lagi: {LLM_MODEL_FILE}")
else:
    # download ~5,8 GB sekali per session
    model_path = hf_hub_download(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE)
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=LLM_N_CTX, n_batch=512, verbose=False)
    print(f"LLM lokal siap: {LLM_MODEL_FILE} (GPU: {llama_cpp.llama_supports_gpu_offload()})")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): reconstructing file:   0%|          |  0.00B / 5.76GB            

gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


LLM lokal siap: gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf (GPU: True)


# Prompt & fungsi LLM lokal


In [17]:
SENTIMENTS = ("positif", "negatif", "netral", "kontroversi")
STANCES = ("oposisi", "moderat", "supporter")
UNCLASSIFIED = "tidak_terklasifikasi"
KOL_FIELDS = ["relevan", "summary", "sentiment", "stance", "alasan"]
KOL_SCHEMA = {
    "type": "object",
    "properties": {"relevan": {"type": "boolean"},
                   "summary": {"type": "string"},
                   "sentiment": {"type": "string", "enum": list(SENTIMENTS)},
                   "stance": {"type": "string", "enum": list(STANCES)},
                   "alasan": {"type": "string"}},
    "required": KOL_FIELDS,
}

KOL_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}

Cuitan dari akun "{author}" tentang tema ini:
{posts}

Tugas:
1. relevan: true kalau cuitan benar-benar membahas topik di atas; false kalau membahas peristiwa lain
   dan hanya kebetulan menyebut nama tokoh, negara, atau acara yang sama.
2. summary: 2-3 kalimat tentang pandangan atau narasi utama akun ini soal tema tersebut.
   Jelaskan isinya, jangan menyalin cuitan kata per kata.
3. sentiment: SIKAP akun terhadap topik/tema, BUKAN nada bahasanya:
   - positif: mendukung, memuji, atau membela
   - negatif: mengkritik, menolak, atau menyindir
   - netral: hanya menyampaikan informasi, tanpa sikap yang jelas
   - kontroversi: dukungan dan kritik sama kuat
   Contoh: membela Prabowo sambil memaki pengkritiknya = positif (kasar ke pihak lain tidak mengubah sikapnya).
4. stance: sikap akun terhadap {target}, pilih satu:
   - supporter: mendukung, memuji, atau membela {target}
   - oposisi: mengkritik, menolak, atau menyerang {target}
   - moderat: hanya menyampaikan informasi, seimbang, atau kritik yang tidak menolak {target}
   Sentimen positif terhadap tema belum tentu berarti mendukung {target}, dan sebaliknya.
5. alasan: 1-2 kalimat alasan sentiment dan stance, berdasarkan isi cuitan.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def kol_posts(key, theme):
    # post asli akun ini di tema ini, engagement paling gede duluan. kalau nggak ada, pake teks yang nge-RT dia
    own = (df_original[(df_original["author_key"] == key) & (df_original["theme"] == theme)]
           .sort_values("engagement", ascending=False)["text_clean"].dropna().tolist())
    return (own or rt_posts.get((key, theme), []))[:MAX_POSTS_PER_AUTHOR_SUMMARY]


def fmt_posts(posts, max_words=120):
    return "\n".join(f"{i}. " + " ".join(str(p).split()[:max_words]) for i, p in enumerate(posts, 1))


def pick(value, allowed):
    # label dari LLM dirapiin, di luar daftar jadi tidak_terklasifikasi
    value = str(value or "").strip().lower()
    return value if value in allowed else UNCLASSIFIED


def ask_llm(content, schema):
    # satu request ke LLM lokal, balikin dict sesuai schema. dipake lagi di Step 7
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],   # Gemma nggak punya role 'system'
        response_format={"type": "json_object", "schema": schema},
        temperature=0.2, max_tokens=700,
    )
    return json.loads(r["choices"][0]["message"]["content"])


def judge_kol(row, posts):
    # nilai satu KOL: relevan, summary, sentiment, stance, alasan
    out = ask_llm(KOL_PROMPT.format(topic=TOPIC, theme=row["theme"].replace("_", " "), target=STANCE_TARGET,
                                    author=row["author_id"], posts=fmt_posts(posts)), KOL_SCHEMA)
    return {"relevan": bool(out.get("relevan", True)), "summary": str(out.get("summary") or "").strip(),
            "sentiment": pick(out.get("sentiment"), SENTIMENTS), "stance": pick(out.get("stance"), STANCES),
            "alasan": str(out.get("alasan") or "").strip()}

# Gemma nilai tiap KOL


In [18]:
LLM_TEST_N = None   # isi angka buat nyoba beberapa KOL dulu, None = semua KOL

kol_texts = {}   # post yang dibaca LLM per KOL, dipake lagi di Step 6C
records = []
todo = df_top_authors if LLM_TEST_N is None else df_top_authors.head(LLM_TEST_N)
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Sentimen + stance + summary (LLM lokal)"):
    posts = kol_posts(row["Id"], row["theme"])
    if not posts:   # KOL yang cuma muncul di jaringan, nggak punya teks
        continue
    kol_texts[(row["Id"], row["theme"])] = posts
    try:
        out = judge_kol(row, posts)
    except Exception as e:
        print(f"Error di {row['author_id']} ({row['theme']}): {type(e).__name__}: {e}")
        out = {"relevan": None, "summary": "", "sentiment": UNCLASSIFIED, "stance": UNCLASSIFIED,
               "alasan": f"error LLM: {type(e).__name__}"}
    records.append({**row[["theme", "rank", "author_id", "Id", "score"]].to_dict(),
                    "jumlah_teks": len(posts), **out})   # jumlah cuitan yang dibaca LLM

df_kol = pd.DataFrame(records)
print(f"KOL yang dinilai: {len(df_kol)}, dilewatin karena nggak ada teks: {len(todo) - len(df_kol)}")
off_topic = df_kol["relevan"].eq(False)
if off_topic.any():
    print(f"{off_topic.sum()} KOL ternyata nggak bahas topik (salah tema di Step 4): "
          f"{', '.join(df_kol.loc[off_topic, 'author_id'].astype(str))}")
    print("KOL ini nggak dihitung di hasil akhir")
if LLM_TEST_N is not None:
    print(f"Baru nyoba {len(df_kol)} KOL. Kalau udah oke, set LLM_TEST_N = None terus jalanin ulang")

Sentimen + stance + summary (LLM lokal):   0%|          | 0/18 [00:00<?, ?it/s]

KOL yang dinilai: 12, dilewatin karena nggak ada teks: 6


# Validasi Gemini


In [19]:
def build_validation_prompt(rows):
    # cuitan pake "-" biar nomornya cuma buat akun (field index nggak ketuker)
    posts = lambda r: "\n".join("     - " + " ".join(str(p).split()[:120]) for p in kol_texts[(r.Id, r.theme)])
    items = "\n\n".join(
        f'{i}. Akun "{r.author_id}" | tema: {r.theme.replace("_", " ")}\n'
        f"   Cuitan:\n{posts(r)}\n"
        f"   Hasil model lokal -> sentiment: {r.sentiment}, stance: {r.stance}. Alasan: {r.alasan}"
        for i, r in enumerate(rows.itertuples(), 1)
    )
    return f"""Kamu validator analisis opini publik media sosial Indonesia. Topik: {TOPIC}.
Pihak yang dinilai untuk stance: {STANCE_TARGET}.

Definisi:
- sentiment = SIKAP akun terhadap topik/tema (bukan nada bahasa): positif, negatif, netral, kontroversi.
  Membela tokoh sambil memaki pengkritiknya tetap positif.
- stance = sikap terhadap {STANCE_TARGET}: supporter (mendukung/membela), oposisi (mengkritik/menolak),
  moderat (informatif, seimbang, atau kritik yang tidak menolak).

Periksa hasil model lokal untuk tiap akun berdasarkan cuitannya. Kalau sudah benar, ulangi labelnya;
kalau salah, ganti dengan label yang benar.

{items}

Jawab HANYA JSON array berisi tepat {len(rows)} objek dengan field:
index (nomor akun), sentiment, stance."""


def apply_validation(rows, results):
    # masukin hasil gemini ke df_kol, label yang nggak valid tetep pake label Gemma
    for (df_idx, row), r in zip(rows.iterrows(), results):
        sent = r.get("sentiment") if r.get("sentiment") in SENTIMENTS else row["sentiment"]
        stance = r.get("stance") if r.get("stance") in STANCES else row["stance"]
        changed = (sent, stance) != (row["sentiment"], row["stance"])
        df_kol.loc[df_idx, ["sentiment_final", "stance_final", "validasi"]] = [
            sent, stance, "dikoreksi" if changed else "setuju"]


df_kol["sentiment_final"] = df_kol["sentiment"]   # default pake hasil Gemma, ditimpa kalau dikoreksi gemini
df_kol["stance_final"] = df_kol["stance"]
df_kol["validasi"] = "belum_divalidasi"

to_check = df_kol[df_kol["relevan"].ne(False) & df_kol["sentiment"].ne(UNCLASSIFIED)]
quota_hit = False
for b in tqdm(range(0, len(to_check), BATCH_SIZE_VALIDATE), desc="Validasi Gemini"):
    rows = to_check.iloc[b:b + BATCH_SIZE_VALIDATE]
    try:
        results = ask_json(build_validation_prompt(rows), parse=lambda p: order_by_index(p, len(rows)), retry=3)
    except Exception as e:
        if not is_quota_error(e):
            raise
        quota_hit = True
        break
    if results is not None:
        apply_validation(rows, results)
    time.sleep(SLEEP_BETWEEN_CALLS)

counts = df_kol["validasi"].value_counts()
print(f"Validasi gemini: {counts.get('setuju', 0)} setuju, {counts.get('dikoreksi', 0)} dikoreksi, "
      f"{counts.get('belum_divalidasi', 0)} belum divalidasi")
print(f"Total request AI: {api_call_count}")
if quota_hit:
    print("Kuota Colab AI habis, KOL yang belum divalidasi pake hasil Gemma. "
          "Tunggu beberapa menit terus jalanin ulang cell ini")

Validasi Gemini:   0%|          | 0/3 [00:00<?, ?it/s]

Validasi gemini: 0 setuju, 0 dikoreksi, 12 belum divalidasi
Total request AI: 8
Kuota Colab AI habis, KOL yang belum divalidasi pake hasil Gemma. Tunggu beberapa menit terus jalanin ulang cell ini


# Ringkasan per tema

In [20]:
THEME_FIELDS = ["ringkasan", "narasi_supporter", "narasi_moderat", "narasi_oposisi"]
THEME_SCHEMA = {"type": "object", "properties": {f: {"type": "string"} for f in THEME_FIELDS}, "required": THEME_FIELDS}

THEME_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}
Komposisi sikap KOL di tema ini: {composition}

Ringkasan tiap KOL, urut dari yang paling berpengaruh:
{kols}

Tugas (HANYA berdasarkan ringkasan di atas, jangan menambah fakta dari luar):
1. ringkasan: 3-4 kalimat tentang apa yang dibicarakan para KOL di tema ini dan ke mana arah percakapannya.
2. narasi_supporter: 1-2 kalimat argumen utama kubu supporter. Tulis "-" kalau tidak ada supporter.
3. narasi_moderat: 1-2 kalimat argumen utama kubu moderat. Tulis "-" kalau tidak ada.
4. narasi_oposisi: 1-2 kalimat argumen utama kubu oposisi. Tulis "-" kalau tidak ada.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def summarize_theme(theme, kols):
    # satu request per tema, dari summary KOL (bukan cuitan mentah) biar prompt-nya nggak kepanjangan
    counts = kols["stance_final"].value_counts()
    content = THEME_PROMPT.format(
        topic=TOPIC, theme=theme.replace("_", " "), target=STANCE_TARGET,
        composition=", ".join(f"{s} {counts.get(s, 0)}" for s in STANCES),
        kols="\n".join(f"- [{r.stance_final}] {r.author_id}: {r.summary}" for r in kols.itertuples()),
    )
    out = ask_llm(content, THEME_SCHEMA)
    return {f: str(out.get(f) or "").strip() for f in THEME_FIELDS}


theme_records = []
usable = df_kol[(df_kol["summary"] != "") & df_kol["relevan"].ne(False)]   # KOL yang salah tema nggak ikut
for theme, kols in usable.sort_values("rank").groupby("theme"):
    try:
        out = summarize_theme(theme, kols)
    except Exception as e:
        print(f"Error di tema {theme}: {type(e).__name__}: {e}")
        out = dict.fromkeys(THEME_FIELDS, "")
    counts = kols["stance_final"].value_counts()
    theme_records.append({"theme": theme, "jumlah_kol": len(kols),
                          **{s: int(counts.get(s, 0)) for s in STANCES}, **out})

df_theme_summary = pd.DataFrame(theme_records)
print(f"Ringkasan {len(df_theme_summary)} tema selesai")
if LLM_TEST_N is not None:
    print("Ini masih dari KOL mode tes. Set LLM_TEST_N = None di Step 6B-2, jalanin ulang, baru cell ini")

Ringkasan 3 tema selesai


In [21]:
print(f"KOL per tema (stance terhadap {STANCE_TARGET}):")
display(df_kol[df_kol["relevan"].ne(False)]
        [["theme", "rank", "author_id", "score", "jumlah_teks", "sentiment_final", "stance_final", "summary"]]
        .rename(columns={"theme": "tema", "author_id": "akun", "score": "skor",
                         "sentiment_final": "sentimen", "stance_final": "stance"}))

print("\nRingkasan per tema:")
display(df_theme_summary.rename(columns={"theme": "tema"}))

print(f"\nNodes: {len(df_nodes)} akun-tema, 30 teratas menurut pagerank:")
display(df_nodes.sort_values("pagerank", ascending=False).head(30)
        [["theme", "akun", "pagerank", "eigenvector", "followers", "is_kol"]].rename(columns={"theme": "tema"}))

print(f"\nEdges: {len(df_edges)}, 30 teratas menurut weight:")
display(df_edges.sort_values("Weight", ascending=False).head(30).rename(columns={"theme": "tema"}))

KOL per tema (stance terhadap Presiden Prabowo dan pemerintah):


,tema,rank,akun,skor,jumlah_teks,sentimen,stance,summary
0,agenda_pertemuan,1,@admkuznetsov45,0.834274,1,netral,moderat,"Akun ini menilai pemerintah Indonesia telah melakukan banyak langkah untuk ketahanan energi, seperti program konversi minyak sawit menjadi diesel B50, kontrak impor minyak mentah dari Rusia, dan mempertimbangkan impor lebih banyak minyak mentah dari negara-negara Afrika."
1,agenda_pertemuan,2,@morinagan0tchil,0.500698,1,netral,moderat,Akun ini mempertanyakan mengapa Prabowo tidak melakukan kunjungan ke Rusia saat kunjungannya ke Vladivostok.
2,kerja_sama_bilateral,1,@Fatimahbnda,0.425027,1,positif,supporter,"Akun ini positif terhadap kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, karena melihat peluang kerja sama di berbagai sektor, termasuk pengembangan pabrik urea di Vladivostok."
3,kerja_sama_bilateral,2,Bahlil Lahadalia,0.375000,1,positif,supporter,Bahlil Lahadalia menyatakan bahwa Rusia telah menjadi mitra penting bagi Indonesia selama lebih dari 5 dasawarsa dan memiliki peluang besar untuk memperkuat konektivitas dan kerja sama yang menguntungkan kedua negara.
4,kerja_sama_bilateral,3,@szerlyjesline,0.300130,1,positif,supporter,"Akun ini mengapresiasi pertemuan Prabowo-Putin yang membahas peluang investasi di sektor pupuk, energi, dan perkapalan, termasuk penjajakan pabrik urea di Vladivostok."
5,kerja_sama_bilateral,4,@lbug9390,0.300028,1,netral,moderat,"Akun ini menyampaikan bahwa pertemuan Prabowo dan Putin membahas peluang kerja sama investasi di berbagai sektor, termasuk penjajakan pembangunan pabrik urea di Vladivostok."
6,kerja_sama_bilateral,5,@idnscope_,0.300007,1,positif,supporter,"Akun ini membahas pertemuan Prabowo dan Putin yang membahas peluang investasi di berbagai sektor, termasuk penjajakan pabrik pupuk urea di Vladivostok."
7,kerja_sama_bilateral,6,Auto Populer ID,0.300000,1,positif,supporter,"Akun ini memuji Prabowo karena memperkuat modernisasi militer Indonesia dengan membeli alutsista strategis dari India dan Rusia, termasuk rudal BrahMos dan Astra. Akun ini juga menanyakan dampak pembelian ini terhadap kekuatan militer Indonesia dan bagaimana negara lain memandang Indonesia."
8,reaksi_geopolitik,1,kliper pemerintah,0.300000,2,positif,supporter,Akun ini memuji Prabowo Subianto karena dianggap memiliki strategi geopolitik yang kuat dan berani menghadapi Amerika Serikat.
9,reaksi_geopolitik,2,Meta AI,0.225000,1,netral,moderat,"Cuitan ini membahas klaim Presiden Prabowo tentang ketersediaan solar di dunia yang ketat, dan menjelaskan bahwa klaim tersebut hiperbola untuk mendorong kemandirian B50."



Ringkasan per tema:


,tema,jumlah_kol,oposisi,moderat,supporter,ringkasan,narasi_supporter,narasi_moderat,narasi_oposisi
0,agenda_pertemuan,2,0,2,0,"Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin. Mereka juga membahas agenda pertemuan, termasuk ketahanan energi Indonesia dan kemungkinan impor minyak mentah dari Rusia dan Afrika.",-,"KOL moderat mempertanyakan agenda pertemuan dan langkah-langkah yang diambil pemerintah Indonesia untuk ketahanan energi, seperti program konversi minyak sawit menjadi diesel B50 dan kontrak impor minyak mentah dari Rusia.",-
1,kerja_sama_bilateral,6,0,1,5,"Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, dengan fokus pada peluang kerja sama bilateral di berbagai sektor seperti pupuk, energi, dan perkapalan. Percakapan ini cenderung positif dan optimis tentang potensi kerja sama yang menguntungkan kedua negara.","Para KOL supporter memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, melihatnya sebagai peluang untuk memperkuat kerja sama bilateral di berbagai sektor, termasuk pengembangan pabrik urea di Vladivostok.","KOL moderat menyampaikan bahwa pertemuan Prabowo dan Putin membahas peluang kerja sama investasi di berbagai sektor, termasuk penjajakan pembangunan pabrik urea di Vladivostok.",-
2,reaksi_geopolitik,4,0,2,2,"Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin, serta reaksi geopolitik atas kunjungan tersebut. Percakapan berfokus pada strategi geopolitik Prabowo, klaim ketersediaan solar, dan upaya memperkuat hubungan bilateral dengan Rusia.","Kubu supporter memuji Prabowo Subianto karena dianggap memiliki strategi geopolitik yang kuat dan berani menghadapi Amerika Serikat, serta mendukung kunjungan ke Rusia sebagai upaya memperkuat geopolitik dan bilateral.","Kubu moderat membahas klaim ketersediaan solar yang hiperbola untuk mendorong kemandirian B50, serta membagikan artikel tentang pernyataan Presiden RI di EEF 2026 di Vladivostok, Rusia.",-



Nodes: 8 akun-tema, 30 teratas menurut pagerank:


,tema,akun,pagerank,eigenvector,followers,is_kol
1,agenda_pertemuan,@mhuseinali,0.278654,0.353553,24541.0,True
0,agenda_pertemuan,@admkuznetsov45,0.246230,0.707107,1158.0,True
6,agenda_pertemuan,ukraine,0.098838,0.353553,NaN,True
5,agenda_pertemuan,cingkonoha,0.098838,0.353553,NaN,True
3,agenda_pertemuan,@BabiNgepet01nk,0.069360,0.353553,127.0,True
2,agenda_pertemuan,@AlQossam3,0.069360,0.353553,1894.0,True
4,agenda_pertemuan,@morinagan0tchil,0.069360,0.500000,116.0,True
7,agenda_pertemuan,@PelawiSamuel,0.069360,0.353553,707.0,True



Edges: 6, 30 teratas menurut weight:


,Source,Target,interaction,tema,Weight
0,admkuznetsov45,mhuseinali,reply,agenda_pertemuan,1
1,alqossam3,admkuznetsov45,retweet,agenda_pertemuan,1
2,babingepet01nk,admkuznetsov45,retweet,agenda_pertemuan,1
3,morinagan0tchil,cingkonoha,reply,agenda_pertemuan,1
4,morinagan0tchil,ukraine,mention,agenda_pertemuan,1
5,pelawisamuel,admkuznetsov45,retweet,agenda_pertemuan,1
